In [1]:
# python
# Remember to patch numba by changing at AppData\Local\Programs\Python\Python312\Lib\site-packages\numba\__init__.py the line 
# if numpy_version > (2, 2): by if numpy_version > (2, 3):
import pickle
import numpy as np
import random
from scipy.sparse import csr_matrix

# AutDEC
from autdec.bb_cln import *
from autdec.perm_utils import perm_mat_from_aut
from autdec.igraph_auts import *

#Extra
import os
import boto3
import io
import pickle
import logging
from botocore.exceptions import ClientError

/Users/pabloezquerromoya/UCM/Master/CuanticaDistribuida/.venv/lib/python3.12/site-packages/numba/cpython/hashing.py:477: UserWarning: FNV hashing is not implemented in Numba. See PEP 456 https://www.python.org/dev/peps/pep-0456/ for rationale over not using FNV. Numba will continue to work, but hashes for built in types will be computed using siphash24. This will permit e.g. dictionaries to continue to behave as expected, however anything relying on the value of the hash opposed to hash as a derived property is likely to not work as expected.
  warnings.warn(msg)


In [2]:
n = 144

For the [[72,12,6]], [[90,8,10]] and [[144,12,12]] BB codes, the graph automorphism files of the detector error model check matrix are pregenerated and stored in the `bivariate_bicycle_codes\graph_auts` folder. 

For the rest of the BB codes in the original paper, the detector error model graph automorphisms can be computed below by entering the corresponding physical qubit number `n`.

In [3]:
# Load auts
if n == 72: 
    code_name = 'BB72'
    chk_shape = (252,2232)
    k = 12
    with open(f'graph_auts/n{n}k{k}_graph_autsZ.pkl', 'rb') as f:
        auts = pickle.load(f)
        auts_ord = len(auts)
    DEM_col_perms = [np.eye(chk_shape[1],dtype=int)]
    DEM_row_perms = [np.eye(chk_shape[0],dtype=int)]
    for a in auts: 
        P = perm_mat_from_aut(a,chk_shape[0]+chk_shape[1])
        DEM_col_perms.append(P[:chk_shape[1],:chk_shape[1]])
        DEM_row_perms.append(P[chk_shape[1]:,chk_shape[1]:])
elif n==90:
    code_name = 'BB90'
    chk_shape = (495,4590)
    k = 8
    with open(f'graph_auts/n{n}k{k}_graph_autsZ.pkl', 'rb') as f:
        auts = pickle.load(f)
        auts_ord = len(auts)
    DEM_col_perms = [np.eye(chk_shape[1],dtype=int)]
    DEM_row_perms = [np.eye(chk_shape[0],dtype=int)]
    for a in auts: 
        P = perm_mat_from_aut(a,chk_shape[0]+chk_shape[1])
        DEM_col_perms.append(P[:chk_shape[1],:chk_shape[1]])
        DEM_row_perms.append(P[chk_shape[1]:,chk_shape[1]:])
elif n==144:
    code_name = 'BB144'
    chk_shape = (936,8784)
    k = 12
    with open(f'graph_auts/n{n}k{k}_graph_autsZ.pkl', 'rb') as f:
        auts = pickle.load(f)
        auts_ord = len(auts)
    DEM_col_perms = [np.eye(chk_shape[1],dtype=int)]
    DEM_row_perms = [np.eye(chk_shape[0],dtype=int)]
    for a in auts: 
        P = perm_mat_from_aut(a,chk_shape[0]+chk_shape[1])
        DEM_col_perms.append(P[:chk_shape[1],:chk_shape[1]])
        DEM_row_perms.append(P[chk_shape[1]:,chk_shape[1]:])

else: 
    code_name = f'BB{n}'
    chk = bb_dem_matrix(code_name) # DEM check matrix
    chk_shape = chk.shape
    DEM_col_perms, DEM_row_perms = vertex_graph_auts_from_bliss(chk,print_order=False)
    auts_ord=len(DEM_row_perms)
print(f"Automorphisms: {auts_ord+1}")


Automorphisms: 72


## Choose ensemble size

In [4]:
# Pick random elements from list of automorphisms
batchsize = 72
auts_ind_list = list(range(1, batchsize))
auts_ind_list=np.hstack((0,auts_ind_list)) # always include identity perm (base decoder)


DEM_col_perms = [DEM_col_perms[a] for a in auts_ind_list]
DEM_row_perms = [DEM_row_perms[a] for a in auts_ind_list]
print(f'No of auts taken: {len(auts_ind_list)}')

No of auts taken: 72


# Generation of the matrices

In [5]:
    bb_code_name = 'BB144'
    d = 12
    basis = 'Z'
    error_rate = 0.0030
    
    if bb_code_name == 'BB72' or bb_code_name == 'bb72':
        # [[72,12,6]]
        code, A_list, B_list = create_bivariate_bicycle_codes(6, 6, [3], [1,2], [1,2], [3])
        d = 6
    elif bb_code_name == 'BB90' or bb_code_name == 'bb90':
        # [[90,8,10]]
        code, A_list, B_list = create_bivariate_bicycle_codes(15, 3, [9], [1,2], [2,7], [0])
        d = 10
    elif bb_code_name == 'BB108' or bb_code_name == 'bb108':
        # [[108,8,10]]
        code, A_list, B_list = create_bivariate_bicycle_codes(9, 6, [3], [1,2], [1,2], [3])
        d = 10
    elif bb_code_name == 'BB144' or bb_code_name == 'bb144':
        # [[144,12,12]]
        code, A_list, B_list = create_bivariate_bicycle_codes(12, 6, [3], [1,2], [1,2], [3])
        d = 12
    elif bb_code_name == 'BB288' or bb_code_name == 'bb288':
        # [[288,12,18]]
        code, A_list, B_list = create_bivariate_bicycle_codes(12, 12, [3], [2,7], [1,2], [3])
        d = 18
    elif bb_code_name == 'BB360' or bb_code_name == 'bb360':
        # [[360,12,<=24]]
        code, A_list, B_list = create_bivariate_bicycle_codes(30, 6, [9], [1,2], [25,26], [3])
        d = 24
    elif bb_code_name == 'BB756' or bb_code_name == 'bb756':
        # [[756,16,<=34]]
        code, A_list, B_list = create_bivariate_bicycle_codes(21,18, [3], [10,17], [3,19], [5])
        d = 34
    else: 
        raise ValueError('Not a valid BB code.')
    
    

    if basis == 'Z':
        z_basis = True
    elif basis == 'X':
        z_basis = False

    ## DEM 
    circuit = build_circuit(code, A_list, B_list, 
                            p=error_rate, # physical error rate
                            num_repeat=d, # usually set to code distance
                            z_basis=z_basis,   # whether in the z-basis or x-basis
                            use_both=False, # whether use measurement results in both basis to decode one basis
                            )
    dem = circuit.detector_error_model()

    chk, obs, priors, col_dict = dem_to_check_matrices(dem, return_col_dict=True)

In [6]:
DEM_priors_list = []
DEM_ensemble_list = []
no_of_auts = len(DEM_col_perms)
for i in range(no_of_auts): 
    col_perms = DEM_col_perms[i]
    new_checks = chk@col_perms
    DEM_ensemble_list.append(new_checks) #List of detector error model automorphisms
    new_priors = priors @ col_perms 
    DEM_priors_list.append(new_priors)  #List of priors according to the corresponding detector error model automorphism   

---
S3

In [ ]:
def get_connection_s3():
    return boto3.client(
        's3',
        endpoint_url=os.getenv('S3_ENDPOINT_URL'),
        region_name=os.getenv('AWS_DEFAULT_REGION'),
        aws_access_key_id=os.getenv('AWS_ACCESS_KEY_ID'),
        aws_secret_access_key=os.getenv('AWS_SECRET_ACCESS_KEY')
    )

# Function to upload
def upload_automorphisms_to_s3(DEM_ensemble_list, DEM_priors_list, DEM_row_perms, error_rate, code_name='BB144'):
    """
    Upload each automorphism to S3 in the structure:
    automorphisms/error_rate/auto_<id>/
    """
    s3 = get_connection_s3()
    
    # Ensure error_rate is formatted correctly for path
    error_rate_str = f"{error_rate:.6f}".rstrip('0').rstrip('.')
    
    for idx in range(len(DEM_ensemble_list)):
        # Create data dictionary for this automorphism
        auto_data = {
            'ensemble': DEM_ensemble_list[idx],
            'priors': DEM_priors_list[idx],
            'row_perm': DEM_row_perms[idx]
        }
        # Define S3 path for this automorphism
        s3_path = f"automorphisms/{error_rate_str}/auto_{idx}/data.pkl"
        
        try:
            s3.put_object(
                Bucket=os.getenv('S3_BUCKET_NAME'),
                Key=s3_path,
                Body=pickle.dumps(auto_data)
            )
            logging.info(f"Uploaded automorphism {idx} to s3://{os.getenv('S3_BUCKET_NAME')}/{s3_path}")
        except ClientError as e:
            logging.error(f"Error uploading automorphism {idx}: {str(e)}")
            raise

# Function to download a specific automorphism
def download_automorphism_from_s3(auto_id, error_rate):
    """
    Download a specific automorphism from S3
    """
    s3 = get_connection_s3()
    error_rate_str = f"{error_rate:.6f}".rstrip('0').rstrip('.')
    s3_path = f"automorphisms/{error_rate_str}/auto_{auto_id}/data.pkl"
    
    try:
        response = s3.get_object(
            Bucket=os.getenv('S3_BUCKET_NAME'),
            Key=s3_path
        )
        data = pickle.loads(response['Body'].read())
        return data['ensemble'], data['priors'], data['row_perm']
    except ClientError as e:
        logging.error(f"Error downloading automorphism {auto_id}: {str(e)}")
        return None, None, None

In [9]:
upload_automorphisms_to_s3(
    DEM_ensemble_list=DEM_ensemble_list,
    DEM_priors_list=DEM_priors_list,
    DEM_row_perms=DEM_row_perms,
    error_rate=error_rate
)

In [ ]:
# Download a specific automorphism (example for auto_0)
ensemble, priors, row_perm = download_automorphism_from_s3(auto_id=0, error_rate=error_rate)


In [ ]:
# Check types and shapes
# print(f"\nType of first element in DEM_ensemble_list: {type(DEM_ensemble_list[0])}")
# print(f"Shape of first element in DEM_ensemble_list: {DEM_ensemble_list[0].shape}")
# print(DEM_ensemble_list[0])
# print(f"Type of saved ensemble: {type(ensemble)}")
# print(f"Shape of ensemble {ensemble.shape}")
# print(ensemble)

print(f"\nType of first element in DEM_priors_list: {type(DEM_priors_list[0])}")
print(f"Shape of first element in DEM_priors_list: {DEM_priors_list[0].shape}")
print(DEM_priors_list[0])
print(f"Type of saved priors: {type(priors)}")
print(f"Shape of priors {priors.shape}")
print(priors)

# print(f"\nType of first element in DEM_row_perms: {type(DEM_row_perms[0])}")
# print(f"Shape of first element in DEM_row_perms: {DEM_row_perms[0].shape}")
# print(f"Type of saved row_perm: {type(row_perm)}")
# print(f"Shape of row_perm {row_perm.shape}")
# print(row_perm)

---

In [ ]:
#################SERIAL#################

In [ ]:
import numpy as np

def ldpc_dec_msaa_quantum_serial(llr, Nloop, Hrows, Hcols, alpha, Syn_x, random_order):
    # Get the size of the parity check matrix representations
    rHR, cHR = Hrows.shape
    rHC, cHC = Hcols.shape
    
    
    # Initialize variables
    letar     = np.zeros((rHR, cHR))
    letac     = np.zeros((rHC, cHC))
    etar      = np.zeros((rHR, cHR))
    etar_old  = np.zeros((rHR, cHR))
    etac      = np.zeros((rHC, cHC))
    Hdec      = np.zeros((Nloop + 1, llr.shape[0]), dtype=int)

    # Input saturation
    llr = np.clip(llr, -10000, 10000)
    lambda_ = llr.copy()

    # Loop through all iterations
    for loop in range(Nloop + 1):
        row_indices = np.random.permutation(rHR) if random_order else range(rHR)
        for irHR in range(rHR):
            min1 = np.inf
            min2 = np.inf
            sign_t = 1

            # Check node update
            for icHR in range(cHR):
                idx = Hrows[irHR, icHR]
                if idx == 0:
                    break

                idx -= 1  # Convert to 0-based index
                lambda_[idx] -= etar_old[irHR, icHR]

                abs_val = abs(lambda_[idx])
                if abs_val < min1:
                    min2 = min1
                    min1 = abs_val
                    posm = icHR
                elif abs_val < min2:
                    min2 = abs_val

                if lambda_[idx] < 0:
                    sign_t = -sign_t

            for icHR in range(cHR):
                idx = Hrows[irHR, icHR]
                if idx == 0:
                    break

                idx -= 1
                pr = (min2 if icHR == posm else min1) * alpha
                sign = sign_t
                if lambda_[idx] < 0:
                    sign = -sign

                etar[irHR, icHR] = sign * pr * Syn_x[irHR]

            for icHR in range(cHR):
                idx = Hrows[irHR, icHR]
                if idx == 0:
                    break

                idx -= 1
                lambda_[idx] += etar[irHR, icHR]
                Hdec[loop, idx] = int(lambda_[idx] <= 0)

        etar_old = etar.copy()

        # Parity check
        parity_t = 0
        for irHR in range(rHR):
            sign_t = 0
            for icHR in range(cHR):
                idx = Hrows[irHR, icHR]
                if idx == 0:
                    break
                idx -= 1
                sign_t += Hdec[loop, idx]

            if sign_t % 2 != ((Syn_x[irHR] - 1) // -2):
                parity_t += 1

        if parity_t == 0:
            break

    return Hdec, loop, lambda_


In [ ]:
import numpy as np

def build_Hcols_Hrows(H):
    H = np.asarray(H)

    # Hcols: nonzero row indices per column
    max_col_nnz = max(np.count_nonzero(H[:, i]) for i in range(H.shape[1]))
    Hcols = np.zeros((H.shape[1], max_col_nnz), dtype=int)
    for i in range(H.shape[1]):
        nz_indices = np.flatnonzero(H[:, i]) + 1  # +1 to match MATLAB indexing
        Hcols[i, :len(nz_indices)] = nz_indices

    # Hrows: nonzero column indices per row
    max_row_nnz = max(np.count_nonzero(H[i, :]) for i in range(H.shape[0]))
    Hrows = np.zeros((H.shape[0], max_row_nnz), dtype=int)
    for i in range(H.shape[0]):
        nz_indices = np.flatnonzero(H[i, :]) + 1
        Hrows[i, :len(nz_indices)] = nz_indices

    return Hcols, Hrows

In [ ]:
Hcols_list = []
Hrows_list = []
for en_n in range(18):
    Hcols, Hrows = build_Hcols_Hrows(DEM_ensemble_list[en_n])
    Hcols_list.append(Hcols)
    Hrows_list.append(Hrows)

In [ ]:
PlBPserial = 0
max_iter = 30
scaling_factors = [1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00]
for i in range(5000000):
    #Noise model
    sampler = circuit.compile_detector_sampler()
    num_shots = 1
    detectors, observables = sampler.sample(num_shots, separate_observables=True)
    #Decoder. NOTE that we need to be consistent with the permutations of the automorphism for the order of the detectors
    # Run decoder multiple times
    candidates = []
    for en_n in range(18):
        detectors_en = DEM_row_perms[en_n]@detectors[0]%2
        detectors_bin = 1 - 2 * detectors_en.astype(int)
        channel_probs = DEM_priors_list[en_n]
        llr = np.log((1 - channel_probs) / channel_probs)
        pred_err, iters, soft_info = ldpc_dec_msaa_quantum_serial(llr, max_iter, Hrows_list[en_n], Hcols_list[en_n], scaling_factors[en_n], detectors_bin, True)
        # Only consider results that finished before max_iter
        if iters < max_iter:
            error_weight = np.sum(pred_err@llr)
            candidates.append((error_weight, pred_err, iters, soft_info))
    # Select the one with the fewest errors
    if candidates:
        best_error_weight, best_pred_err, best_iters, best_soft_info = min(candidates, key=lambda x: x[0])
        pred_err = best_pred_err
        iters = best_iters
        soft_info = best_soft_info
    else:
        # Fallback: none finished early, use the first run or handle as needed
        detectors_en = DEM_row_perms[en_n]@detectors[0]%2
        detectors_bin = 1 - 2 * detectors_en.astype(int)
        channel_probs = DEM_priors_list[en_n]
        llr = np.log((1 - channel_probs) / channel_probs)
        pred_err, iters, soft_info = ldpc_dec_msaa_quantum_serial(llr, max_iter, Hrows, Hcols, 1.00, detectors_bin, True)
                
    
    logical_error_osd = (obs@pred_err[iters]+observables) % 2

    #print('Logical errors:')
    #print(logical_error_osd)
    #print('Detectors:')
    #print(detectors[0])

    if np.any(logical_error_osd == 1):
        PlBPserial = PlBPserial + 1  
        print(PlBPserial/i)
print(f'Error BPOSD: {PlBPserial}')

# Pruebas Pablo

In [70]:
from ldpc import BpDecoder

# Test the decoder used in adapted_test_v2.py
PlBP_ldpc = 0

test_pcm = chk
test_error_channel = priors
test_row_perm = DEM_row_perms[0]

_bp = BpDecoder(
    test_pcm, 
    max_iter=100, 
    ms_scaling_factor=0.9,
    error_rate=float(0.003), 
    bp_method="minimum_sum", 
    error_channel=test_error_channel
)

print(f"Testing decoder from adapted_test_v2.py...")

for i in range(1000):  # Test with 1000 samples
    # Generate noise
    sampler = circuit.compile_detector_sampler(seed=42 + i + 1)  # Seed for reproducibility
    num_shots = 1
    detectors, observables = sampler.sample(num_shots, separate_observables=True)
    
    # Apply automorphism transformation
    transformed_detectors = (test_row_perm @ detectors[0] % 2)
    
    # Decode
    predicted_error = _bp.decode(transformed_detectors)
    
    # Check logical error
    logical_error = (obs @ predicted_error + observables) % 2
    
    if np.any(logical_error == 1):
        PlBP_ldpc += 1
        print(f"Logical error corrected in pattern {i+1}")

print(f"\nFinal results:")
print(f"Total patterns: 1000")
print(f"Logical errors: {PlBP_ldpc}")
print(f"Logical error rate: {PlBP_ldpc/1000:.6f}")

Testing decoder from adapted_test_v2.py...
Logical error corrected in pattern 2
Logical error corrected in pattern 6
Logical error corrected in pattern 20
Logical error corrected in pattern 32
Logical error corrected in pattern 37
Logical error corrected in pattern 46
Logical error corrected in pattern 61
Logical error corrected in pattern 64
Logical error corrected in pattern 86
Logical error corrected in pattern 94
Logical error corrected in pattern 96
Logical error corrected in pattern 100
Logical error corrected in pattern 105
Logical error corrected in pattern 120
Logical error corrected in pattern 129
Logical error corrected in pattern 142
Logical error corrected in pattern 147
Logical error corrected in pattern 150
Logical error corrected in pattern 154
Logical error corrected in pattern 156
Logical error corrected in pattern 168
Logical error corrected in pattern 178
Logical error corrected in pattern 180
Logical error corrected in pattern 186
Logical error corrected in pattern

In [ ]:
from ldpc import BpDecoder

# Test the decoder used in adapted_test_v2.py
PlBP_ldpc = 0

test_pcm = chk
test_error_channel = priors

_bp = BpDecoder(
    test_pcm, 
    max_iter=100, 
    ms_scaling_factor=0.9,
    error_rate=float(0.003), 
    bp_method="minimum_sum", 
    error_channel=test_error_channel
)

print(f"Testing decoder from adapted_test_v2.py...")

for i in range(1000):  # Test with 1000 samples
    # Generate noise
    sampler = circuit.compile_detector_sampler(seed=42 + i + 1)  # Seed for reproducibility
    num_shots = 1
    detectors, observables = sampler.sample(num_shots, separate_observables=True)
        
    # Decode
    predicted_error = _bp.decode(detectors[0])
    
    # Check logical error
    logical_error = (obs @ predicted_error + observables) % 2
    
    if np.any(logical_error == 1):
        PlBP_ldpc += 1
        print(f"Logical error corrected in pattern {i+1}")

print(f"\nFinal results:")
print(f"Total patterns: 1000")
print(f"Logical errors: {PlBP_ldpc}")
print(f"Logical error rate: {PlBP_ldpc/1000:.6f}")

Testing decoder from adapted_test_v2.py...
Logical error corrected in pattern 2
Logical error corrected in pattern 6
Logical error corrected in pattern 20
Logical error corrected in pattern 32
Logical error corrected in pattern 37
Logical error corrected in pattern 46
Logical error corrected in pattern 61
Logical error corrected in pattern 64
Logical error corrected in pattern 86
Logical error corrected in pattern 94
Logical error corrected in pattern 96
Logical error corrected in pattern 100
Logical error corrected in pattern 105
Logical error corrected in pattern 120
Logical error corrected in pattern 129
Logical error corrected in pattern 142
Logical error corrected in pattern 147
Logical error corrected in pattern 150
Logical error corrected in pattern 154
Logical error corrected in pattern 156
Logical error corrected in pattern 168
Logical error corrected in pattern 178
Logical error corrected in pattern 180
Logical error corrected in pattern 186
Logical error corrected in pattern

## Diferencia en la manera de obtener datos relacionados con las matrices entre NB y AppCloud

### dem_error_channel

In [68]:
# Notebook
chk, obs, priors, col_dict = dem_to_check_matrices(dem, return_col_dict=True)
print(priors[8063])

0.00917250560204902


In [ ]:
# AppCloud

from dem_to_matrices import detector_error_model_to_check_matrices

matrices = detector_error_model_to_check_matrices(dem, allow_undecomposed_hyperedges=True)
print(matrices.priors[8063])

0.009131148241452289


#### PCM igual con ambas funciones

In [ ]:
# Comparación de matrices chk(Notebook) <-> matrices.check_matrix(AppCloud)
if(np.array_equal(chk.toarray(), matrices.check_matrix.toarray())):
    print("PCM iguales")

PCM iguales
